# Itertools => Recipes, Patterns & Best Practices

## Recipes: Building Your Own Tools

The tools in `itertools` are building blocks. The official documentation shows many **recipes** that combine them. Their main purpose is educational.

Many recipes are also available ready-made in the third-party package `more-itertools`.

## Everyday Recipes

| Recipe | Idea |
|---|---|
| `take(n, it)` | `list(islice(it, n))` |
| `nth(it, n)` | `next(islice(it, n, None), default)` |
| `flatten(lists)` | `chain.from_iterable(lists)` |
| `chunked(it, n)` | `batched(it, n)` |
| `sliding_window(it, n)` | Overlapping windows of size `n` |
| `unique_everseen(it)` | Remove duplicates, keep order |
| `roundrobin(*its)` | Take one item from each iterable in turn |
| `powerset(it)` | All subsets, using `combinations` |
| `first_true(it, pred)` | First item where the predicate is true |
| `quantify(it, pred)` | Count how many items pass a test |

## Pipeline Style

Combine small lazy steps. Each step consumes the previous one.

```python
lines = ["  10 ", "x", "20", "", "30"]

numbers = (int(s) for s in map(str.strip, lines) if s.isdigit())
running = accumulate(numbers)
list(running)      # [10, 30, 60]
```

## Comprehension, Generator, `itertools` or Loop?

| Situation | Choose |
|---|---|
| Simple transform or filter that builds a collection | Comprehension |
| One pass over a large or unknown-size stream | Generator expression |
| Chain, slice, group, batch, window, combine | `itertools` |
| Side effects, complex branching, several steps of state | Plain `for` loop |

## Pitfalls Checklist

- **Exhausted iterators:** a second pass over the same iterator gives nothing. Store the data in a `list` if you need it twice.
- **Infinite iterators:** always limit them.
- **`groupby` on unsorted data:** creates many small groups.
- **`tee` memory:** the buffered data can grow large.
- **`zip` truncation:** use `zip(strict=True)` or `zip_longest`.
- **Late binding:** lazy pipelines read variables when they run, not when you built them.

```python
threshold = 2
big = filter(lambda n: n > threshold, [1, 2, 3, 4])
threshold = 3
list(big)          # [4]  (the lambda saw threshold = 3)
```

## Performance Notes

- The tools are implemented in C, so combining them is often faster than an equivalent Python-level loop.
- Lazy pipelines save memory because no full intermediate list is built.
- Readability still comes first. Measure with `timeit` before rewriting working code.

## Best Practices

- Build pipelines from small, named steps.
- Convert to `list` only at the end, or when you need to reuse the data.
- Keep infinite iterators behind `islice` or `takewhile`.
- Prefer `chain.from_iterable` over sum-of-lists tricks.
- Reach for `more-itertools` when a needed recipe is missing.

## Source

https://docs.python.org/3/library/itertools.html#itertools-recipes

https://pypi.org/project/more-itertools/

In [ ]:
from collections import deque
from itertools import accumulate, chain, combinations, count, cycle, filterfalse, islice

# --- Recipes built from the basic tools ---
def take(n, iterable):
    return list(islice(iterable, n))

def nth(iterable, n, default=None):
    return next(islice(iterable, n, None), default)

def flatten(list_of_lists):
    return chain.from_iterable(list_of_lists)

def sliding_window(iterable, n):
    iterator = iter(iterable)
    window = deque(islice(iterator, n - 1), maxlen=n)
    for item in iterator:
        window.append(item)
        yield tuple(window)

def unique_everseen(iterable):
    seen = set()
    for item in filterfalse(seen.__contains__, iterable):
        seen.add(item)
        yield item

def roundrobin(*iterables):
    iterators = [iter(it) for it in iterables]
    while iterators:
        for iterator in list(iterators):
            try:
                yield next(iterator)
            except StopIteration:
                iterators.remove(iterator)

def powerset(iterable):
    items = list(iterable)
    return chain.from_iterable(combinations(items, r) for r in range(len(items) + 1))

def first_true(iterable, default=False, predicate=None):
    return next(filter(predicate, iterable), default)

def quantify(iterable, predicate=bool):
    return sum(map(predicate, iterable))

print(take(3, count(10)))                                # [10, 11, 12]
print(nth("ABCDEF", 3))                                  # D
print(list(flatten([[1, 2], [3], [4, 5]])))              # [1, 2, 3, 4, 5]
print(list(sliding_window("ABCDE", 3)))                  # overlapping windows
print(list(unique_everseen("AAABBCAD")))                 # ['A', 'B', 'C', 'D']
print(list(roundrobin("ABC", "D", "EF")))                # ['A', 'D', 'E', 'B', 'F', 'C']
print(list(powerset([1, 2, 3])))
print(first_true([0, "", None, 7, 9]))                   # 7
print(quantify([1, 5, 8, 2], lambda n: n > 3))           # 2

# --- Pipeline style: small lazy steps ---
lines = ["  10 ", "x", "20", "", "30"]
numbers = (int(s) for s in map(str.strip, lines) if s.isdigit())
print(list(accumulate(numbers)))                         # [10, 30, 60]

# --- Pitfall: a second pass over an iterator is empty ---
stream = iter([1, 2, 3])
print(sum(stream), sum(stream))                          # 6 0

# --- Pitfall: late binding in lazy pipelines ---
threshold = 2
big = filter(lambda n: n > threshold, [1, 2, 3, 4])
threshold = 3
print(list(big))                                         # [4]: the lambda saw threshold = 3

# --- Pitfall: zip stops at the shortest ---
print(list(zip("ABC", range(2))))                        # [('A', 0), ('B', 1)]

# --- A round-robin schedule with cycle ---
workers = cycle(["w1", "w2", "w3"])
tasks = ["t1", "t2", "t3", "t4", "t5"]
print([(task, next(workers)) for task in tasks])